# Tarea: Geometría de Datos en Producción
## Escalamiento y Discretización (Google Play Store)

**Objetivo:**
En la clase anterior demostramos cómo las variables con escalas gigantescas secuestran el espacio vectorial y destruyen los algoritmos de Machine Learning. Ahora es su turno de aplicar estos conceptos arquitectónicos al *dataset* de la Google Play Store que limpiaron la semana pasada.

**Instrucciones:**
Resuelvan los siguientes 4 retos programando directamente en las celdas de código de este Notebook. Para cada reto, respondan a la pregunta analítica utilizando un comentario o una celda Markdown.

In [1]:
import pandas as pd
import numpy as np

# Instrucción: Carga tu archivo limpio de la semana 4 ('playstore_limpio.csv').
# Si por alguna razón no lo tienes, usa el siguiente bloque para descargar y pre-limpiar lo básico:

url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"
df = pd.read_csv(url)

# Limpieza rápida (Mockup del resultado de la Semana 4)
df.drop_duplicates(subset=['App'], inplace=True)
df = df[df['Installs'] != 'Free'] # Quitar la fila de error ("Life Made WI-Fi")
df['Installs'] = df['Installs'].str.replace('+', '', regex=False).str.replace(',', '', regex=False).astype(float)
df['Reviews'] = df['Reviews'].astype(float)
df['Rating'] = df['Rating'].astype(float)
# Rellenar ratings nulos con la mediana para que el escalamiento no falle
df['Rating'] = df['Rating'].fillna(df['Rating'].median())

print("Datos listos. Total de apps:", len(df))

Datos listos. Total de apps: 9659


---
### Reto 1: La Trampa de Min-Max en un Mundo Viral
La columna `Installs` (Descargas) representa un reto enorme porque vivimos en un mundo asimétrico: hay millones de apps con 10 descargas y unas cuantas con 1,000 millones (como WhatsApp).

1. Crea una nueva columna llamada `Installs_MinMax` aplicando la fórmula $(X - Min) / (Max - Min)$.
2. Ejecuta un `.describe()` sobre tu nueva columna.
3. **Pregunta de Análisis:** Observa el percentil 75% (`75%`) de tu nueva columna comprimida. ¿Qué valor numérico tiene? Como arquitecto de datos, explica qué nos dice este número sobre el riesgo de usar *Min-Max Scaler* en variables virales (con *outliers* masivos).

In [2]:
# 1. Aplica la fórmula Min-Max a la columna 'Installs'
df['Installs_MinMax'] = (df['Installs'] - df['Installs'].min()) / (df['Installs'].max() - df['Installs'].min())

# 2. Muestra el resumen estadístico
print(df['Installs_MinMax'].describe())

print("\nApps con Installs_MinMax > 0.5:", (df['Installs_MinMax'] > 0.5).sum())

count    9659.000000
mean        0.007778
std         0.053758
min         0.000000
25%         0.000001
50%         0.000100
75%         0.001000
max         1.000000
Name: Installs_MinMax, dtype: float64

Apps con Installs_MinMax > 0.5: 20


### RESPUESTA A LA PREGUNTA (Escribe aquí tu análisis):

_El percentil 75% dio **0.001**. O sea que el 75% de las apps quedaron aplastadas entre 0 y 0.001, practicamente en cero, y la mediana quedó todavia peor (0.0001)._

_Esto pasa porque Min-Max toma el valor máximo (1,000 millones de descargas, tipo WhatsApp) como el 1.0, y todo lo demás se mide contra ese monstruo. Una app con 100,000 descargas, que en la vida real ya es bastante, queda en 0.0001. Solo 20 apps de las 9,659 quedaron arriba de 0.5._

_El riesgo es que para el algoritmo casi todas las apps se ven iguales (todas valen ~0) y las unicas que "existen" en el espacio vectorial son los outliers. Se pierde toda la información del resto de las descargas. Con variables virales asi, lo que conviene es aplicar primero `np.log1p()` como vimos la clase pasada, o usar un escalador robusto que no dependa del máximo._

---
### Reto 2: Estandarización de Calificaciones (Z-Score)
La columna `Rating` va de 1.0 a 5.0. Vamos a estandarizarla para que su media sea 0 y su desviación estándar sea 1 (Campana de Gauss). En una distribución normal, cualquier valor con un $Z-Score$ menor a $-3$ es estadísticamente una "anomalía negativa" extrema.

1. Crea la columna `Rating_Z` aplicando la fórmula $(X - \mu) / \sigma$.
2. Escribe una máscara booleana para filtrar el *DataFrame* y encontrar cuántas aplicaciones tienen un `Rating_Z` menor a `-3.0`.
3. **Pregunta de Análisis:** ¿Cuántas aplicaciones son "anomalías negativas" extremas? ¿Qué calificación original (en estrellas) tenían esas apps para merecer ese Z-Score?

In [3]:
# 1. Aplica la fórmula Z-Score a la columna 'Rating'
df['Rating_Z'] = (df['Rating'] - df['Rating'].mean()) / df['Rating'].std()

# 2. Filtra las apps con Z < -3.0
anomalias_negativas = df[ df['Rating_Z'] < -3.0 ]
print("Total de anomalías negativas:", len(anomalias_negativas))
print(anomalias_negativas[['App', 'Rating', 'Rating_Z']].head())

print("\nRating máximo de las anomalías:", anomalias_negativas['Rating'].max())
print("Rating mínimo de las anomalías:", anomalias_negativas['Rating'].min())
print("Umbral en estrellas para Z = -3:", df['Rating'].mean() - 3 * df['Rating'].std())

Total de anomalías negativas: 194
                                                App  Rating  Rating_Z
477                                      Calculator     2.6 -3.208001
518                   Just She - Top Lesbian Dating     1.9 -4.618162
520  EliteSingles – Dating for Single Professionals     2.5 -3.409453
527                          Sugar Daddy Dating App     2.5 -3.409453
611                                     MouseMingle     2.7 -3.006549

Rating máximo de las anomalías: 2.7
Rating mínimo de las anomalías: 1.0
Umbral en estrellas para Z = -3: 2.703251067741039



### RESPUESTA A LA PREGUNTA:

_Salieron **194 apps** con Z-Score menor a -3.0._

_La media del Rating es como 4.19 estrellas y la desviación estándar 0.50, entonces el umbral de Z = -3 cae en **2.70 estrellas**. Las apps que quedaron como anomalías negativas tienen calificaciones de **2.7 hacia abajo** (la peor es de 1.0). O sea, en la Play Store la gente califica alto casi todo, asi que con tener menos de 2.7 estrellas ya eres una rareza estadística._

---
### Reto 3: Categorización Comercial (El Cuchillo Lógico `pd.cut`)
El equipo de marketing no entiende de estrellas decimales (ej. 4.3). Quieren que las aplicaciones se clasifiquen en 4 etiquetas simples para el *frontend* de la tienda.

1. Utiliza `pd.cut()` sobre la columna original `Rating` para crear una nueva columna llamada `Categoria_Calidad`.
2. Utiliza los siguientes límites (`bins`): `0.0, 3.0, 4.0, 4.5, 5.0`.
3. Asigna las siguientes etiquetas (`labels`): `'Pésima', 'Regular', 'Buena', 'Excelente'`.
4. Ejecuta un `.value_counts()` para ver cuántas hay de cada una. ¿Quedaron desbalanceadas?

In [4]:
# 1 y 2. Aplica pd.cut con los bins y labels especificados
df['Categoria_Calidad'] = pd.cut(df['Rating'],
                                 bins=[0.0, 3.0, 4.0, 4.5, 5.0],
                                 labels=['Pésima', 'Regular', 'Buena', 'Excelente'])

# 4. Muestra los conteos
print(df['Categoria_Calidad'].value_counts())

Categoria_Calidad
Buena        5534
Regular      2063
Excelente    1702
Pésima        360
Name: count, dtype: int64


### RESPUESTA:

_Si, quedaron muy desbalanceadas. "Buena" tiene 5,534 apps (más de la mitad), "Regular" 2,063, "Excelente" 1,702 y "Pésima" nada más 360._

_Es lo normal con `pd.cut`: los cortes los pone uno a mano donde tiene sentido para negocio, no donde están los datos. Como casi todas las apps se concentran entre 4.0 y 4.5, ese grupo se lleva todo. Para marketing sirve, pero para un modelo no._

---
### Reto 4: Cuartiles de *Engagement* (El Cuchillo Estadístico `pd.qcut`)
A diferencia del equipo de marketing, tu modelo de Machine Learning necesita grupos balanceados para no desarrollar sesgos. Vamos a clasificar las aplicaciones basándonos en su volumen de reseñas (`Reviews`).

1. Utiliza `pd.qcut()` sobre la columna `Reviews` para crear 4 grupos de igual tamaño (`q=4`). Llama a la nueva columna `Nivel_Engagement`.
2. Asigna las etiquetas: `'Bajo', 'Medio', 'Alto', 'Viral'`.
3. Ejecuta un `.value_counts()` sobre `Nivel_Engagement`.
4. **Pregunta de Análisis:** Compara este conteo con el que obtuviste en el Reto 3. ¿Por qué el algoritmo de clasificación preferiría predecir la columna `Nivel_Engagement` en lugar de la columna `Categoria_Calidad`?

In [5]:
# 1 y 2. Aplica pd.qcut para generar 4 categorías de igual tamaño
df['Nivel_Engagement'] = pd.qcut(df['Reviews'], q=4, labels=['Bajo', 'Medio', 'Alto', 'Viral'])

# 3. Muestra los conteos (Deberían ser casi idénticos en cada categoría)
print(df['Nivel_Engagement'].value_counts())

# Extra: dónde quedaron los cortes de cada cuartil
print("\nCortes de Reviews por cuartil:")
print(df['Reviews'].quantile([0, 0.25, 0.5, 0.75, 1.0]))

Nivel_Engagement
Bajo     2432
Viral    2415
Alto     2412
Medio    2400
Name: count, dtype: int64

Cortes de Reviews por cuartil:
0.00           0.0
0.25          25.0
0.50         967.0
0.75       29401.0
1.00    78158306.0
Name: Reviews, dtype: float64


#### RESPUESTA A LA PREGUNTA:

_Aqui los 4 grupos quedaron casi iguales: Bajo 2,432, Viral 2,415, Alto 2,412 y Medio 2,400. Comparado con el Reto 3 donde una sola categoría se llevaba el 57%, esto está mucho más parejo._

_El algoritmo preferiria `Nivel_Engagement` porque con `Categoria_Calidad` puede hacer trampa: si predice "Buena" para todo le atina más de la mitad de las veces sin aprender nada, y de las "Pésimas" casi ni ve ejemplos para aprender. Con `pd.qcut` los cortes se ponen en los cuartiles de los datos (25, 967 y 29,401 reseñas), asi que cada clase tiene la misma cantidad de ejemplos y el modelo se ve obligado a encontrar patrones de verdad en lugar de irse con la mayoría._